# SmolVLA × LIBERO-Plus 追加学習（Colab）

teleop で集めた教師データ（LeRobot 形式で Hugging Face に置いたもの）で **SmolVLA を追加学習し、
LIBERO-Plus のシミュレータで動かして動画に残す** notebook。

```text
teleop で操作・記録（Mac） ─▶ LeRobot データセットにして HF へ ─▶ この notebook
                                                                  ├ 追加学習（LoRA / フル）
                                                                  ├ 学習済みモデルをまとめる（LoRA はマージ）
                                                                  ├ HF へアップロード（任意）
                                                                  └ LIBERO-Plus で動かして動画を録る
```

**上から順に実行する。書き換えるのは「1. 設定」のセルだけ。**

既定では、動作確認のために公開データ `lerobot/libero_plus` から 1 タスク分の episode を選んで学習する。
teleop のデータセットができたら `DATASET_REPO` を差し替える。

## 動画のモデルを作ったときの条件（parc2026）

| | |
| :--- | :--- |
| 初期重み | [`lerobot/smolvla_libero_plus`](https://huggingface.co/lerobot/smolvla_libero_plus) @ `7bb70aa5`（VLM は `HuggingFaceTB/SmolVLM2-500M-Video-Instruct`） |
| 学習コード | LeRobot v0.6.0 の `lerobot-train` |
| 教師データ | 人の teleop 18 episode（赤いマグを皿から戻すタスク 1 つ、1,942 frame。非公開）。画像は 128×128 |
| 学習 | LoRA（r 16、α 16）。対象は Action Expert と VLM の言語層の `q_proj`・`v_proj`、state・action の射影層。1,000 step、batch 8、warmup 100 step、lr 1e-4 → 1e-5（cosine）、seed 42、bf16 |
| 最適化 | AdamW（β 0.9 / 0.95、weight decay 1e-10）、勾配クリップ 10（LeRobot の SmolVLA の既定値のまま） |
| GPU | Colab L4。学習 676 秒、最終 loss 0.345 |
| 推論 | 1 回の推論で出す 50 step 分の action のうち、先頭 10 を実行する |

この notebook の学習の既定値（step・batch・学習率・LoRA）は、この条件と同じにしてある。

## 用意するもの

| | |
| :--- | :--- |
| GPU | ランタイム → ランタイムのタイプを変更 → GPU。L4 / A100 を推奨。T4 でも LoRA なら動く（fp16 になる） |
| Hugging Face token | 左の鍵アイコン（Secrets）に `HF_TOKEN` として登録する。アップロードするなら書き込み権限つき |
| Weights & Biases | 任意。使うなら `USE_WANDB = True` にして、Secrets に `WANDB_API_KEY` を登録する |

## データセットの条件

`lerobot/libero_plus` と同じ形式にする。teleop の録画から作るときもこれに合わせる。

| key | 型 | 内容 |
| :--- | :--- | :--- |
| `observation.images.front` | video 256×256×3 | `agentview` カメラ。**env の生画像を 180° 回転**（上下・左右とも反転）したもの |
| `observation.images.wrist` | video 256×256×3 | `robot0_eye_in_hand` カメラ。同じく 180° 回転 |
| `observation.state` | float32 (8,) | eef 位置 3 + eef 姿勢（クォータニオン → 軸角）3 + グリッパ指の位置 2 |
| `action` | float32 (7,) | OSC_POSE の差分 6 + グリッパ 1（teleop の `actions` そのまま） |
| `task` | str | 指示文（teleop の録画 JSON の `language`） |

fps は 20（LIBERO の control_freq と同じ）。

画像の一辺は、`lerobot/libero_plus` と、このリポジトリの `dataset/` が書き出すデータでは 256。
**動画のモデルの教師データは 128×128 だった。** 128 のデータで学習するときは「1. 設定」で `IMAGE_PX = 128` にする
（どちらの解像度でも、SmolVLA はモデルの中で pad 付きで 512×512 に拡大して読む）。

## 1. 設定

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- データ
# teleop のデータセットを HF に上げたら、その repo に差し替える（revision は None で最新）
DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = "f3f49f426d75030177b18778374005bc12ccd588"
# 学習に使う episode。None なら下の絞り込み、絞り込みも None なら全 episode
DATASET_EPISODES = None
# 指示文に含まれる文字列で episode を絞る（lerobot/libero_plus は 40 タスク・14,347 本あるので必ず絞る）
DATASET_TASK_FILTER = "pick up the black bowl between the plate and the ramekin and place it on the plate"
MAX_EPISODES = 50
# 画像の一辺（px）。lerobot/libero_plus と dataset/ の出力は 256、動画のモデルの教師データは 128
IMAGE_PX = 256

# ---------------------------------------------------------------- 初期重み
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = "7bb70aa5bc92b82c9239142775d3a173103567ff"
VLM_REPO = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

# ---------------------------------------------------------------- 学習
# "lora": Action Expert と VLM の言語層に LoRA を付けて学習し、最後に元の重みへマージする（軽い）
# "full": Action Expert 全体を学習する（VLM は凍結）
TRAIN_MODE = "lora"
STEPS = 1000
BATCH_SIZE = 8
SAVE_FREQ = 1000          # この step ごとに checkpoint を保存する
LOG_FREQ = 50
SEED = 42
LORA_R = 16
LORA_ALPHA = 16
# LoRA を付ける層（正規表現）。Action Expert と VLM の言語層の q_proj・v_proj、state・action の射影層。
# None なら LeRobot の既定（VLM の言語層を含まない）
LORA_TARGET_MODULES = (
    r"(model\.vlm_with_expert\.lm_expert\..*\.(q|v)_proj"
    r"|.*\.vlm\.model\.text_model\..*\.self_attn\.(q|v)_proj"
    r"|model\.(state_proj|action_in_proj|action_out_proj|action_time_mlp_in|action_time_mlp_out))"
)
LEARNING_RATE = 1e-4 if TRAIN_MODE == "lora" else 2e-5
FINAL_LEARNING_RATE = LEARNING_RATE / 10
WARMUP_STEPS = 100
USE_WANDB = False
WANDB_PROJECT = "smolvla-libero-finetune"
RUN_NAME = f"smolvla_{TRAIN_MODE}_s{STEPS}_b{BATCH_SIZE}"

# ---------------------------------------------------------------- 保存先
WORK_DIR = Path("/content/work")
OUTPUT_DIR = WORK_DIR / "outputs" / RUN_NAME      # lerobot-train の出力（checkpoint）
FINAL_MODEL_DIR = WORK_DIR / "models" / RUN_NAME  # 推論にそのまま使える形のモデル
PUSH_TO_HUB = False
HF_MODEL_REPO = "your-hf-name/smolvla-libero-finetuned"   # PUSH_TO_HUB = True のとき
HF_MODEL_PRIVATE = True

# ---------------------------------------------------------------- 評価（動画）
# task id は LIBERO-Plus の suite 内の番号（teleop の選択画面・録画 JSON の task_id と同じ）
EVAL_SUITE = "libero_spatial"
EVAL_TASK_IDS = [0]
EVAL_INIT_INDICES = [0, 1, 2]
EVAL_MAX_STEPS = None     # None なら suite の既定（spatial 220 / object 280 / goal 300 / 10 520）
N_ACTION_STEPS = 10       # 1 回の推論で出す 50 step 分の action のうち、実行する数
EVAL_BASELINE = True      # 追加学習前のモデルも同じ条件で動かして比べる
CAMERA_PX = IMAGE_PX      # 学習データと同じ解像度で描画する
VIDEO_DIR = WORK_DIR / "videos" / RUN_NAME

# ---------------------------------------------------------------- 版の固定
LEROBOT_TAG = "v0.6.0"
LIBERO_PLUS_GIT = "https://github.com/sylvestf/LIBERO-plus.git"
LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_ASSETS_REPO = "Sylvest/LIBERO-plus"   # assets.zip（約 6.4 GB、展開後 約 9 GB）

assert TRAIN_MODE in ("lora", "full"), TRAIN_MODE
print(f"run: {RUN_NAME}")

## 2. ランタイムを確認する

In [ ]:
import os
import subprocess
import sys

import torch

if sys.version_info < (3, 12):
    raise RuntimeError(f"LeRobot {LEROBOT_TAG} は Python 3.12 以上が必要です（今は {sys.version.split()[0]}）")
if not torch.cuda.is_available():
    raise RuntimeError("GPU ランタイムを選んでください（ランタイム → ランタイムのタイプを変更）")

os.environ.update({
    "HF_HOME": "/content/hf_cache",
    "HF_LEROBOT_HOME": "/content/lerobot_cache",
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
WORK_DIR.mkdir(parents=True, exist_ok=True)

BF16 = torch.cuda.is_bf16_supported()
MIXED_PRECISION = "bf16" if BF16 else "fp16"
print(f"Python {sys.version.split()[0]}  torch {torch.__version__}  CUDA {torch.version.cuda}")
print(f"GPU    {torch.cuda.get_device_name(0)}  bf16={BF16} -> {MIXED_PRECISION}")
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

## 3. LeRobot を入れる

`lerobot-train` と SmolVLA・PEFT（LoRA）を使うので `training,smolvla,peft` を付けて入れる。
bf16 が使えない GPU（T4 など）では、SmolVLA の VLM を fp16 で読むように 1 行だけ書き換える。

In [ ]:
import importlib
import shutil
from pathlib import Path


def run(command, **kwargs):
    """コマンドを実行し、失敗したら出力の末尾を出して止める。"""
    result = subprocess.run(command, text=True, capture_output=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, command))}\n{(result.stdout + result.stderr)[-4000:]}")
    return result


def replace_once(path, before, after):
    """文字列を 1 か所だけ置き換える。見つからなければ止める（版が変わって素通りするのを防ぐ）。"""
    text = path.read_text(encoding="utf-8")
    if before not in text:
        raise RuntimeError(f"{path.name} に {before!r} が見つかりません。LeRobot の版を確認してください")
    path.write_text(text.replace(before, after, 1), encoding="utf-8")


LEROBOT_DIR = Path("/content/lerobot")
LEROBOT_SRC = LEROBOT_DIR / "src"

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libegl1", "libgl1", "libglib2.0-0", "libmagickwand-dev"])

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)
run(["git", "clone", "--quiet", "--depth", "1", "--branch", LEROBOT_TAG,
     "https://github.com/huggingface/lerobot.git", str(LEROBOT_DIR)])
if not BF16:
    replace_once(LEROBOT_SRC / "lerobot/policies/smolvla/smolvlm_with_expert.py",
                 'torch_dtype="bfloat16",', 'torch_dtype="float16",')
run([sys.executable, "-m", "pip", "install", "--quiet", "-e", f"{LEROBOT_DIR}[training,smolvla,peft]"])

# editable install は次の kernel 起動まで sys.path に載らないので、ここで足す
if str(LEROBOT_SRC) not in sys.path:
    sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

import lerobot  # noqa: E402
import peft  # noqa: E402

print(f"LeRobot {importlib.metadata.version('lerobot')}  ({Path(lerobot.__file__).parent})")
print(f"PEFT    {peft.__version__}")
# LeRobot の依存に合わせて pip が torch を入れ替えると、読み込み済みの torch と食い違う
if importlib.metadata.version("torch").split("+")[0] != torch.__version__.split("+")[0]:
    print(f"WARNING: torch が {torch.__version__} から {importlib.metadata.version('torch')} に入れ替わりました。"
          "ランタイム → セッションを再起動 してから、1. 設定 から順に実行し直してください"
          "（このセルをもう一度実行しても問題ない）")

## 4. Hugging Face にログインする

token は Colab の Secrets（`HF_TOKEN`）から読む。無ければ入力欄が出る（画面には表示されない）。

In [ ]:
from getpass import getpass

from huggingface_hub import login, whoami

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token: ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face: {whoami(token=HF_TOKEN)['name']}")

if USE_WANDB:
    import wandb
    if not os.environ.get("WANDB_API_KEY"):
        try:
            from google.colab import userdata
            os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
        except Exception:
            pass
    if not wandb.login():
        raise RuntimeError("W&B にログインできません。WANDB_API_KEY を確認してください")

## 5. データセットを確かめる

メタ情報だけを読み、形式が SmolVLA × LIBERO-Plus の前提に合っているかと、学習に使う episode を確かめる。

In [ ]:
from collections import Counter

from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata(DATASET_REPO, revision=DATASET_REVISION)
REQUIRED = {
    "observation.images.front": (IMAGE_PX, IMAGE_PX, 3),
    "observation.images.wrist": (IMAGE_PX, IMAGE_PX, 3),
    "observation.state": (8,),
    "action": (7,),
}
problems = [f"{key}: {tuple(meta.features[key]['shape']) if key in meta.features else '無い'} (期待 {shape})"
            for key, shape in REQUIRED.items()
            if key not in meta.features or tuple(meta.features[key]["shape"]) != shape]
if problems:
    raise RuntimeError("データセットの形式が前提と違います:\n  " + "\n  ".join(problems))
if meta.fps != 20:
    print(f"WARNING: fps={meta.fps}（LIBERO の control_freq は 20）")

episodes = meta.episodes.select_columns(["episode_index", "tasks", "length"]).to_pandas()
if DATASET_EPISODES is not None:
    selected = episodes[episodes["episode_index"].isin(DATASET_EPISODES)]
elif DATASET_TASK_FILTER:
    selected = episodes[episodes["tasks"].apply(lambda tasks: any(DATASET_TASK_FILTER in t for t in tasks))]
    selected = selected.head(MAX_EPISODES)
else:
    selected = episodes
if selected.empty:
    raise RuntimeError("学習に使う episode がありません。DATASET_EPISODES / DATASET_TASK_FILTER を確認してください")

# 全 episode を使うときは --dataset.episodes を渡さない（コマンドが長くなりすぎないように）
TRAIN_EPISODES = None if len(selected) == len(episodes) else [int(i) for i in selected["episode_index"]]
train_frames = int(selected["length"].sum())

print(f"dataset   {DATASET_REPO}@{DATASET_REVISION or 'main'}")
print(f"          {meta.total_episodes:,} episodes / {meta.total_frames:,} frames / fps {meta.fps}")
print(f"学習に使う {len(selected):,} episodes / {train_frames:,} frames")
for task, n in Counter(t for tasks in selected["tasks"] for t in tasks).most_common(10):
    print(f"  {n:4d}  {task}")
print(f"学習で見るサンプル数 {STEPS * BATCH_SIZE:,}（各フレーム平均 {STEPS * BATCH_SIZE / train_frames:.1f} 回）")

## 6. 初期重みを取る

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = Path(snapshot_download(
    BASE_MODEL_REPO,
    revision=BASE_MODEL_REVISION,
    allow_patterns=["config.json", "model.safetensors", "train_config.json", "policy_*"],
))
assert (BASE_MODEL_DIR / "model.safetensors").is_file(), BASE_MODEL_DIR
print(f"base model {BASE_MODEL_REPO}@{BASE_MODEL_REVISION}")
print(f"           {BASE_MODEL_DIR}")

## 7. 追加学習する

`lerobot-train` を別プロセスで動かし、`LOG_FREQ` ごとに loss と learning rate を表示する。

- 入出力の形（`input_features` / `output_features`）はデータセットから作り直す。
  正規化の統計量もデータセットのものに置き換わる
- LoRA では元の重みをすべて凍結し、`LORA_TARGET_MODULES` の層に付けた LoRA だけを学習する。
  フル学習では、画像を読む VLM 側を凍結し、行動を出す Action Expert だけを学習する（`train_expert_only`）
- 最初の 1 回はデータセットの動画のダウンロードと読み込みで数分かかる

In [ ]:
import json
import re
from collections import deque

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_DIR}",
    f"--policy.vlm_model_name={VLM_REPO}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    f"--output_dir={OUTPUT_DIR}",
    f"--job_name={RUN_NAME}",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=2",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={SAVE_FREQ}",
    f"--log_freq={LOG_FREQ}",
    f"--wandb.enable={str(USE_WANDB).lower()}",
]
if USE_WANDB:
    command.append(f"--wandb.project={WANDB_PROJECT}")
if DATASET_REVISION:
    command.append(f"--dataset.revision={DATASET_REVISION}")
if TRAIN_EPISODES is not None:
    command.append("--dataset.episodes=" + json.dumps(TRAIN_EPISODES, separators=(",", ":")))
if TRAIN_MODE == "lora":
    command += ["--peft.method_type=LORA", f"--peft.r={LORA_R}", f"--peft.lora_alpha={LORA_ALPHA}"]
    if LORA_TARGET_MODULES:
        command.append(f"--peft.target_modules={LORA_TARGET_MODULES}")

env = dict(os.environ,
           PYTHONPATH=f"{LEROBOT_SRC}{os.pathsep}{os.environ.get('PYTHONPATH', '')}",
           ACCELERATE_MIXED_PRECISION=MIXED_PRECISION,
           PYTHONUNBUFFERED="1",
           TQDM_DISABLE="1")
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"{TRAIN_MODE} / {STEPS} steps / batch {BATCH_SIZE} / lr {LEARNING_RATE:g} -> {FINAL_LEARNING_RATE:g}")
process = subprocess.Popen(command, cwd=LEROBOT_DIR, env=env, text=True, bufsize=1,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
recent = deque(maxlen=60)
losses = []
for line in process.stdout:
    line = line.strip()
    if not line:
        continue
    recent.append(line)
    step = re.search(r"step:\s*([0-9.]+[KMB]?)", line)
    loss = re.search(r"loss:\s*([0-9.eE+-]+)", line)
    if step and loss:
        lr = re.search(r"lr:\s*([0-9.eE+-]+)", line)
        losses.append(float(loss.group(1)))
        print(f"step {step.group(1):>6}  loss {loss.group(1)}  lr {lr.group(1) if lr else '-'}", flush=True)
if process.wait() != 0:
    print("\n".join(recent))
    raise RuntimeError(f"学習が失敗しました（exit {process.returncode}）")
print("学習が終わりました")

## 8. 学習済みモデルをまとめる

LoRA は差分だけが保存されるので、元の重みへマージして普通の SmolVLA モデルにする。
フル学習は最後の checkpoint をそのまま使う。どちらも前処理・後処理（正規化など）の設定を一緒に置く。

In [ ]:
import gc

from safetensors import safe_open

checkpoint_dir = OUTPUT_DIR / "checkpoints" / "last" / "pretrained_model"
if not checkpoint_dir.is_dir():
    checkpoint_dir = OUTPUT_DIR / "checkpoints" / f"{STEPS:06d}" / "pretrained_model"
print(f"checkpoint {checkpoint_dir}")

shutil.rmtree(FINAL_MODEL_DIR, ignore_errors=True)
if TRAIN_MODE == "full":
    shutil.copytree(checkpoint_dir, FINAL_MODEL_DIR)
else:
    from lerobot.configs import PreTrainedConfig
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from peft import PeftModel

    config = PreTrainedConfig.from_pretrained(checkpoint_dir)
    config.device = "cpu"
    config.use_peft = False
    base = SmolVLAPolicy.from_pretrained(BASE_MODEL_DIR, config=config, strict=False)
    merged = PeftModel.from_pretrained(base, checkpoint_dir, is_trainable=False).merge_and_unload(safe_merge=True)
    merged.config.use_peft = False
    merged.config.pretrained_path = None
    merged.config.device = None
    merged.save_pretrained(FINAL_MODEL_DIR)
    for path in checkpoint_dir.glob("policy_*"):
        shutil.copy2(path, FINAL_MODEL_DIR / path.name)
    del base, merged
    gc.collect()
    torch.cuda.empty_cache()

with safe_open(FINAL_MODEL_DIR / "model.safetensors", framework="pt") as weights:
    leftover = [k for k in weights.keys() if "lora_" in k.lower()]
if leftover:
    raise RuntimeError(f"LoRA の重みが残っています: {leftover[:3]}")
for name in sorted(p.name for p in FINAL_MODEL_DIR.iterdir()):
    print(f"  {name}")
print(f"モデル {FINAL_MODEL_DIR}")

## 9. Hugging Face へアップロードする（任意）

`PUSH_TO_HUB = True` のときだけ動く。既定では private の model repo に置く。

In [ ]:
from huggingface_hub import HfApi

if PUSH_TO_HUB:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE, exist_ok=True)
    commit = api.upload_folder(
        folder_path=str(FINAL_MODEL_DIR), repo_id=HF_MODEL_REPO, repo_type="model",
        commit_message=(f"SmolVLA {TRAIN_MODE} fine-tune: {DATASET_REPO}, {STEPS} steps, "
                        f"batch {BATCH_SIZE}, lr {LEARNING_RATE:g} (LeRobot {LEROBOT_TAG})"),
    )
    print(f"https://huggingface.co/{HF_MODEL_REPO}  revision {commit.oid}")
else:
    print("PUSH_TO_HUB = False なのでアップロードしません")

## 10. LIBERO-Plus を準備する

シミュレータ（robosuite / MuJoCo）と LIBERO-Plus 本体、物体やテクスチャの assets を入れる。
assets は約 6.4 GB あり、ダウンロードと展開に数分かかる（2 回目以降は飛ばす）。

Colab では `MUJOCO_GL=egl` だけだと **描画が CPU（llvmpipe）に落ちる**ことがあるので、
NVIDIA の EGL を明示する設定ファイルを作る。実際の描画先は次のセルの最初に `[render]` として表示される。

In [ ]:
import zipfile

from huggingface_hub import hf_hub_download

LIBERO_PLUS_DIR = Path("/content/LIBERO-plus")
LIBERO_ROOT = LIBERO_PLUS_DIR / "libero" / "libero"
ASSETS_DIR = LIBERO_ROOT / "assets"

run([sys.executable, "-m", "pip", "install", "--quiet",
     "robosuite==1.4.1", "mujoco==3.7.0", "bddl==1.0.1", "easydict==1.13", "future",
     "Wand==0.6.13", "scikit-image", "gym==0.26.2", "matplotlib", "PyOpenGL", "imageio[ffmpeg]"])

if not LIBERO_PLUS_DIR.is_dir():
    run(["git", "clone", "--quiet", LIBERO_PLUS_GIT, str(LIBERO_PLUS_DIR)])
run(["git", "-C", str(LIBERO_PLUS_DIR), "checkout", "--quiet", LIBERO_PLUS_SHA])

# LIBERO は初回 import 時にパスを対話で聞くので、先に設定ファイルを書いておく
libero_config = Path.home() / ".libero" / "config.yaml"
libero_config.parent.mkdir(parents=True, exist_ok=True)
libero_config.write_text("".join(f"{key}: {value}\n" for key, value in {
    "benchmark_root": LIBERO_ROOT,
    "bddl_files": LIBERO_ROOT / "bddl_files",
    "init_states": LIBERO_ROOT / "init_files",
    "datasets": LIBERO_ROOT / "datasets",
    "assets": ASSETS_DIR,
}.items()))

if not (ASSETS_DIR / "scenes").is_dir():
    archive = Path(hf_hub_download(LIBERO_PLUS_ASSETS_REPO, "assets.zip", repo_type="dataset",
                                   local_dir="/content/libero_plus_assets"))
    extract_dir = Path("/content/libero_plus_assets/extracted")
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(extract_dir)
    # zip の中で assets/ がどの深さにあっても拾えるように、scenes/ を持つディレクトリを探す
    found = next(p.parent for p in sorted(extract_dir.rglob("scenes"), key=lambda p: len(p.parts)) if p.is_dir())
    shutil.rmtree(ASSETS_DIR, ignore_errors=True)
    shutil.move(str(found), str(ASSETS_DIR))
    archive.unlink()
    shutil.rmtree(extract_dir, ignore_errors=True)
missing = [name for name in ("scenes", "textures", "stable_scanned_objects", "articulated_objects")
           if not (ASSETS_DIR / name).is_dir()]
if missing:
    raise RuntimeError(f"assets が足りません: {missing}")

# NVIDIA の EGL を使わせる（Colab の glvnd には Mesa しか登録されていないことがある）
NVIDIA_LIB = Path("/usr/lib64-nvidia")
egl_icd = WORK_DIR / "10_nvidia_egl.json"
egl_icd.write_text(json.dumps({"file_format_version": "1.0.0",
                               "ICD": {"library_path": str(NVIDIA_LIB / "libEGL_nvidia.so.0")}}))
SIM_ENV = dict(os.environ,
               MUJOCO_GL="egl",
               PYOPENGL_PLATFORM="egl",
               PYTHONPATH=f"{LIBERO_PLUS_DIR}{os.pathsep}{LEROBOT_SRC}",
               PYTHONUNBUFFERED="1")
if (NVIDIA_LIB / "libEGL_nvidia.so.0").exists():
    SIM_ENV["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(egl_icd)
    SIM_ENV["LD_LIBRARY_PATH"] = f"{NVIDIA_LIB}{os.pathsep}{os.environ.get('LD_LIBRARY_PATH', '')}"
print(f"LIBERO-Plus {LIBERO_PLUS_SHA}  assets {ASSETS_DIR}")

## 11. LIBERO-Plus で動かして動画を録る

学習済みモデル（`EVAL_BASELINE = True` なら追加学習前のモデルも）を、同じタスク・同じ初期配置で動かす。
1 回ごとに mp4（左: 正面カメラ、右: 手首カメラ）と結果の 1 行（`results.jsonl`）を残す。

観測は学習データと同じ形にそろえる：画像は 180° 回転、状態は 8 次元、指示文は LIBERO-Plus のタスクの文。
成功はゴール条件（BDDL の `:goal`）の成立で判定する。

In [ ]:
%%writefile /content/work/rollout_libero_plus.py
"""学習した SmolVLA を LIBERO-Plus で動かし、成否と動画を残す。

ノートブックから別プロセスで起動する。MuJoCo の描画設定（EGL）を import より前に
入れるためと、GPU メモリを学習側と分けるため。

観測は学習データ（lerobot/libero_plus と同じ形式）にそろえる。

- 画像: env の生画像を 180° 回転（上下と左右を両方反転）。データセットはこの向きで保存されている
- 状態: [eef 位置 3, eef 姿勢の軸角 3, グリッパ指の位置 2] の 8 次元
- 画像のキー名: データセットと同じ名前で渡す（既定 observation.images.front / wrist）。
  名前の付け替えと正規化は、モデルに同梱の preprocessor が行う
"""

import argparse
import json
import os
import sys
import time
from pathlib import Path

import numpy as np

# suite ごとの既定の最大 step 数（LeRobot の LIBERO 評価と同じ値）
MAX_STEPS = {"libero_spatial": 220, "libero_object": 280, "libero_goal": 300,
             "libero_10": 520, "libero_90": 400}
# 初期配置の直後は物体が落ち着くまで待つ（グリッパを開いたまま何もしない）
NUM_WAIT_STEPS = 10
DUMMY_ACTION = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]


def parse_args():
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", required=True, help="SmolVLA のディレクトリ（model.safetensors と processor を含む）")
    p.add_argument("--label", default="model", help="動画名と結果に付ける名前")
    p.add_argument("--suite", default="libero_spatial", choices=sorted(MAX_STEPS))
    p.add_argument("--task-ids", type=int, nargs="+", default=[0])
    p.add_argument("--init-indices", type=int, nargs="+", default=[0])
    p.add_argument("--max-steps", type=int, default=None, help="省略時は suite ごとの既定値")
    p.add_argument("--n-action-steps", type=int, default=10,
                   help="1 回の推論で出した action 列のうち、実際に実行する数")
    p.add_argument("--camera-px", type=int, default=256, help="env の描画解像度（学習データと同じにする）")
    p.add_argument("--front-key", default="observation.images.front")
    p.add_argument("--wrist-key", default="observation.images.wrist")
    p.add_argument("--seed", type=int, default=7)
    p.add_argument("--device", default=None, help="省略時は cuda → mps → cpu の順で選ぶ")
    p.add_argument("--fps", type=int, default=20)
    p.add_argument("--out", type=Path, required=True)
    return p.parse_args()


def probe_renderer():
    """MuJoCo が実際にどの GPU で描画しているかを返す（Linux のみ）。

    Colab では MUJOCO_GL=egl でも CPU 描画（Mesa / llvmpipe）に落ちることがある。
    落ちても動くが遅いので、警告だけ出して続ける。
    """
    if sys.platform != "linux":
        return None
    import mujoco
    from OpenGL import GL

    model = mujoco.MjModel.from_xml_string("<mujoco><worldbody><geom size='.1'/></worldbody></mujoco>")
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)
    renderer = mujoco.Renderer(model, 8, 8)
    renderer.update_scene(data)
    renderer.render()
    vendor = GL.glGetString(GL.GL_VENDOR).decode()
    name = GL.glGetString(GL.GL_RENDERER).decode()
    renderer.close()
    return f"{vendor} / {name}"


def patch_torch_load():
    """LIBERO の初期状態ファイルは weights_only=False でないと読めない（torch 2.6 以降の既定は True）。"""
    import torch

    original = torch.load

    def load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return original(*args, **kwargs)

    torch.load = load


def state8(obs):
    from robosuite.utils.transform_utils import quat2axisangle

    return np.concatenate([
        np.asarray(obs["robot0_eef_pos"], dtype=np.float64),
        quat2axisangle(np.asarray(obs["robot0_eef_quat"], dtype=np.float64)),
        np.asarray(obs["robot0_gripper_qpos"], dtype=np.float64),
    ]).astype(np.float32)


def upright(image):
    """env の生画像（上下逆さま）を 180° 回して、データセットと同じ向きにする。"""
    return np.flip(np.asarray(image), axis=(0, 1)).copy()


def to_model_input(obs, args, instruction):
    import torch

    def chw(image):
        return torch.from_numpy(upright(image)).permute(2, 0, 1).float().div(255.0)

    return {
        args.front_key: chw(obs["agentview_image"]),
        args.wrist_key: chw(obs["robot0_eye_in_hand_image"]),
        "observation.state": torch.from_numpy(state8(obs)),
        "task": instruction,
    }


def load_policy(args, device):
    from lerobot.policies.factory import make_pre_post_processors
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

    policy = SmolVLAPolicy.from_pretrained(args.model)
    if not 1 <= args.n_action_steps <= policy.config.chunk_size:
        raise SystemExit(f"--n-action-steps は 1..{policy.config.chunk_size} の範囲で指定する")
    policy.config.n_action_steps = args.n_action_steps
    policy.to(device)
    policy.eval()
    preprocessor, postprocessor = make_pre_post_processors(
        policy.config,
        pretrained_path=args.model,
        preprocessor_overrides={"device_processor": {"device": device}},
    )
    expected = list(policy.config.image_features)
    print(f"[policy] {args.model}  image keys={expected}  n_action_steps={args.n_action_steps}", flush=True)
    return policy, preprocessor, postprocessor


def write_video(path, frames, fps):
    import imageio.v2 as imageio

    path.parent.mkdir(parents=True, exist_ok=True)
    with imageio.get_writer(path, fps=fps, codec="libx264", quality=7, macro_block_size=1) as writer:
        for frame in frames:
            writer.append_data(frame)


def run_episode(env, init_state, policy, preprocessor, postprocessor, args, instruction, max_steps):
    import torch

    env.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_WAIT_STEPS):
        obs, _, _, _ = env.step(DUMMY_ACTION)
    policy.reset()

    frames = [np.concatenate([upright(obs["agentview_image"]), upright(obs["robot0_eye_in_hand_image"])], axis=1)]
    success, steps = False, 0
    for steps in range(1, max_steps + 1):
        batch = preprocessor(to_model_input(obs, args, instruction))
        with torch.inference_mode():
            action = policy.select_action(batch)
        action = postprocessor(action)
        action = np.asarray(action.detach().cpu(), dtype=np.float64).reshape(-1)
        if action.shape != (7,) or not np.isfinite(action).all():
            raise RuntimeError(f"action が不正: shape={action.shape}")
        obs, _, done, _ = env.step(action.tolist())
        frames.append(np.concatenate([upright(obs["agentview_image"]), upright(obs["robot0_eye_in_hand_image"])], axis=1))
        if done or env.check_success():
            success = True
            break
    return success, steps, frames


def main():
    args = parse_args()
    renderer = probe_renderer()
    if renderer is not None:
        print(f"[render] {renderer}", flush=True)
        if "llvmpipe" in renderer or "Mesa" in renderer:
            print("[render] WARNING: CPU 描画です。動きますが遅くなります（EGL の設定を確認）", flush=True)

    import torch

    patch_torch_load()
    from libero.libero import benchmark
    from libero.libero.envs import OffScreenRenderEnv

    device = args.device or ("cuda" if torch.cuda.is_available()
                             else "mps" if torch.backends.mps.is_available() else "cpu")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    policy, preprocessor, postprocessor = load_policy(args, device)

    suite = benchmark.get_benchmark_dict()[args.suite](task_order_index=0)
    max_steps = args.max_steps or MAX_STEPS[args.suite]
    args.out.mkdir(parents=True, exist_ok=True)
    results_path = args.out / "results.jsonl"

    results = []
    for task_id in args.task_ids:
        task = suite.get_task(task_id)
        bddl = suite.get_task_bddl_file_path(task_id)
        init_states = suite.get_task_init_states(task_id)
        env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=args.camera_px,
                                 camera_widths=args.camera_px, horizon=max_steps + NUM_WAIT_STEPS + 10,
                                 ignore_done=True)
        env.seed(args.seed)
        print(f"[task] {args.suite} #{task_id}  {task.language}", flush=True)
        for init_index in args.init_indices:
            if not 0 <= init_index < len(init_states):
                print(f"  [skip] init {init_index} は範囲外（0..{len(init_states) - 1}）", flush=True)
                continue
            t0 = time.time()
            success, steps, frames = run_episode(env, init_states[init_index], policy, preprocessor,
                                                 postprocessor, args, task.language, max_steps)
            video = args.out / f"{args.label}_{args.suite}_t{task_id}_i{init_index}_{'success' if success else 'fail'}.mp4"
            write_video(video, frames, args.fps)
            row = dict(label=args.label, model=str(args.model), suite=args.suite, task_id=task_id,
                       task=task.name, instruction=task.language, init_index=init_index,
                       success=success, steps=steps, max_steps=max_steps,
                       n_action_steps=args.n_action_steps, seed=args.seed,
                       seconds=round(time.time() - t0, 1), video=str(video))
            results.append(row)
            with results_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(f"  init {init_index}: {'SUCCESS' if success else 'fail'}  steps={steps}  "
                  f"{row['seconds']}s  -> {video.name}", flush=True)
        env.close()

    n_success = sum(r["success"] for r in results)
    print(f"[done] {args.label}: {n_success}/{len(results)} success", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
def rollout(model_dir, label):
    command = [sys.executable, str(WORK_DIR / "rollout_libero_plus.py"),
               "--model", str(model_dir), "--label", label,
               "--suite", EVAL_SUITE,
               "--task-ids", *map(str, EVAL_TASK_IDS),
               "--init-indices", *map(str, EVAL_INIT_INDICES),
               "--n-action-steps", str(N_ACTION_STEPS),
               "--camera-px", str(CAMERA_PX),
               "--seed", str(SEED),
               "--out", str(VIDEO_DIR)]
    if EVAL_MAX_STEPS:
        command += ["--max-steps", str(EVAL_MAX_STEPS)]
    process = subprocess.Popen(command, env=SIM_ENV, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    recent = deque(maxlen=40)
    for line in process.stdout:
        recent.append(line.rstrip())
        # スクリプト自身の行だけ出す（LIBERO-Plus はタスク番号を数千件並べたログを出す）
        if line.startswith(("[policy]", "[render]", "[task]", "[done]", "  init", "  [skip]")):
            print(line.rstrip(), flush=True)
    if process.wait() != 0:
        print("\n".join(recent))
        raise RuntimeError(f"{label} の rollout が失敗しました（exit {process.returncode}）")


VIDEO_DIR.mkdir(parents=True, exist_ok=True)
(VIDEO_DIR / "results.jsonl").unlink(missing_ok=True)
if EVAL_BASELINE:
    rollout(BASE_MODEL_DIR, "base")
rollout(FINAL_MODEL_DIR, "finetuned")

## 12. 結果と動画を見る

In [ ]:
import pandas as pd
from IPython.display import HTML, Video, display

results = pd.read_json(VIDEO_DIR / "results.jsonl", lines=True)
display(results[["label", "suite", "task_id", "init_index", "success", "steps", "seconds"]])
display(results.groupby("label")["success"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "成功", "count": "試行", "mean": "成功率"}))

for row in results.itertuples():
    display(HTML(f"<b>{row.label}</b> {row.suite} #{row.task_id} init {row.init_index} — "
                 f"{'SUCCESS' if row.success else 'fail'}（{row.steps} steps）<br>{row.instruction}"))
    display(Video(row.video, embed=True, width=512))

動画・結果・学習済みモデルを手元に持ち帰る（Colab の VM は切断すると消える）。

In [ ]:
archive = shutil.make_archive(str(WORK_DIR / f"{RUN_NAME}_videos"), "zip", VIDEO_DIR)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass